# Session 10 — The Training Loop: making a small model tell the truth about itself

Five lines run every training job in the world:

```python
logits, loss = model(x, targets=y)   # forward
loss.backward()                      # fill in every gradient
optimizer.step()                     # move every weight
optimizer.zero_grad()                # wipe before the next batch
```

This notebook takes those lines and **checks them**, because every serious
training bug is silent and the loss curve is not the one that will tell you.

The assignment, point by point:

1. **Print every tensor shape** in the step, and say what each dimension means.
2. **Verify one gradient by hand** — nudge a weight, measure the loss change, compare to `backward()`.
3. **Break gradient accumulation on purpose** — average-of-averages vs. token-weighted, micro-batches of different lengths, both curves on one plot.
4. **Log the grad norm at every step**, then find a step where it moved before the loss did.
5. **Compute your own MFU**, report it honestly, and say what is costing you the distance to 40%.
6. **Take the number 0.1** and write it out in fp32, bf16 and fp8 E4M3, bit by bit. Say which one you would train in.

**Stack:** a `uv`-managed project, a compact nanoGPT-style decoder (~1–15M params),
char-level Tiny Shakespeare, PyTorch. All code lives in `src/trainloop/`.

> **Two scales.** Defaults are a small-GPU proxy. Set `S10_SMOKE=1` before
> launching to shrink the model, data and step counts so the whole notebook runs
> top-to-bottom on a CPU in a couple of minutes — that is what `dry_run.py` uses.


## 0. Setup


In [ ]:
import os, sys, json, math, time
import numpy as np
import torch

sys.path.insert(0, "src")
from trainloop import utils, data, model as model_mod
from trainloop import shapes, gradcheck, accum, gradnorm, mfu, floats

SMOKE = utils.is_smoke()
utils.seed_everything(1337)
DEVICE = utils.pick_device()
print(f"smoke mode : {SMOKE}")
print(f"device     : {DEVICE}  ({utils.device_name(DEVICE)})")
print(f"torch      : {torch.__version__}")

RESULTS = {}


### The model and the data

Char-level Tiny Shakespeare (vocab ≈ 65). A char-level vocab means an untrained
model's loss should sit near `ln(vocab_size)` — a free sanity anchor we use
later.


In [ ]:
ds = data.load_char_dataset(smoke=SMOKE)
print(f"vocab_size      : {ds.vocab_size}")
print(f"ln(vocab_size)  : {math.log(ds.vocab_size):.3f}   <- an untrained model should land here")
print(f"train chars     : {len(ds.train_ids):,}")
print(f"val chars       : {len(ds.val_ids):,}")

if SMOKE:
    cfg = model_mod.GPTConfig(vocab_size=ds.vocab_size, block_size=64,
                              n_layer=2, n_head=2, n_embd=64)
else:
    cfg = model_mod.GPTConfig(vocab_size=ds.vocab_size, block_size=256,
                              n_layer=6, n_head=6, n_embd=384)

net = model_mod.GPT(cfg).to(DEVICE)
print(cfg)
print(f"total params        : {net.num_params(non_embedding=False):,}")
print(f"non-embedding params: {net.num_params(non_embedding=True):,}   (the N in 6N)")
RESULTS["config"] = cfg.__dict__.copy()
RESULTS["params_total"] = net.num_params(non_embedding=False)
RESULTS["params_non_embedding"] = net.num_params(non_embedding=True)


## 1. Print every tensor shape in the step

One forward + backward pass, with the shape of **every** tensor printed and a
one-line meaning for each dimension: the activations flowing through, every
weight tensor, and every gradient (whose shape must equal its parameter's).

The dimension letters:

| letter | meaning |
| --- | --- |
| `B` | batch — independent sequences in this step |
| `T` | time / sequence length — token positions, left to right |
| `C` | channels (`n_embd`) — width of the residual stream |
| `H` | heads — parallel attention subspaces |
| `Dh` | `C / H` — width of one head |
| `F` | `4C` — MLP hidden width |
| `V` | `vocab_size` — number of output symbols |


In [ ]:
xb, yb = data.get_batch(ds, "train", batch_size=4,
                        seq_len=min(16, cfg.block_size), device=DEVICE)
info = shapes.instrument_step(net, xb, yb)


**What each dimension means, in one line each:**

- `idx [B, T]` — B sequences, each T token ids. This is the whole input.
- `token embedding [B, T, C]` — every id becomes a C-dim vector; the table is `[V, C]`.
- `position embedding [T, C]` — one vector per slot 0…T-1, added to every sequence.
- `q,k,v [B, H, T, Dh]` — the stream split into H heads; within a head each of T positions has a Dh query, key and value.
- `attention scores [B, H, T, T]` — for each head, how much every position attends to every position at or before it.
- `attention out / c_proj [B, T, C]` — heads concatenated back to width C and mixed.
- `MLP c_fc [B, T, F]` then `c_proj [B, T, C]` — widen to 4C, non-linearity, project back.
- `logits [B, T, V]` — an unnormalised score for every vocab symbol at every position.
- `per-token loss [B, T]` — cross-entropy at each position; `loss []` — the scalar mean over counted tokens.
- **weights**: `c_attn.weight [3C, C]`, `c_proj.weight [C, C]`, `mlp.c_fc.weight [F, C]`, `mlp.c_proj.weight [C, F]`, LayerNorm `[C]`, embedding/head `[V, C]` (tied).
- **gradients**: identical shape to their weight — `dL/dW` has one number per weight.


## 2. Verify one gradient by hand

Pick one scalar weight `w`. `backward()` reports `dL/dw`. Now measure it the
dumb way — nudge `w` up by `h`, nudge it down by `h`, see how the loss moved:

$$\frac{\partial \mathcal{L}}{\partial w} \approx \frac{\mathcal{L}(w+h) - \mathcal{L}(w-h)}{2h}$$

Done in fp64 with dropout off. They should agree to several decimals.


In [ ]:
res = gradcheck.verify_one_gradient(
    net, xb, yb,
    param_name="transformer.h.0.mlp.c_fc.weight",
    index=(0, 0),
    step_h=1e-4,
)
print(res.render())
assert res.matching_decimals >= 4, "analytic and numeric gradient disagree — investigate!"
RESULTS["gradcheck_analytic"] = res.analytic
RESULTS["gradcheck_numeric"] = res.numeric
RESULTS["gradcheck_matching_decimals"] = res.matching_decimals


If these had *not* agreed, the usual suspects are: dropout still on (forward not
repeatable), a too-large or too-small `h` (truncation vs. rounding), fp32
instead of fp64, or an in-place op corrupting the autograd graph. Agreement to
6+ decimals means `backward()` on this weight is doing exactly the chain rule
you would do by hand.


## 3. Break gradient accumulation on purpose

Combining K micro-batches into one step, two ways:

- **correct** — sum every per-token loss, divide by the total number of tokens. Every token gets one equal vote.
- **wrong** — average each micro-batch's *mean* loss ("average of the averages"). A short micro-batch gets the same vote as a long one.

They're identical when micro-batches hold equal token counts — which is why
this bug lived in every major framework until 2024. First the arithmetic from
the class notes:


In [ ]:
demo = accum.average_of_averages_demo(token_counts=(4, 4, 2), mean_losses=(2.0, 2.0, 5.0))
for k, v in demo.items():
    print(f"{k:>34} : {v}")
RESULTS["accum_static_correct"] = demo["correct (sum loss / sum tokens)"]
RESULTS["accum_static_wrong"] = demo["wrong (mean of means)"]
RESULTS["accum_static_rel_err_pct"] = demo["relative error %"]


Now the same mistake inside a real loop: two copies of the same initial model,
fed the **same** unequal-length micro-batches every step (token counts
`(48, 48, 12)`), one trained with each rule.


In [ ]:
acfg = accum.AccumConfig(
    n_steps=20 if SMOKE else 120,
    real_lens=(48, 48, 12) if not SMOKE else (24, 24, 6),
    pad_to=64 if not SMOKE else 32,
    micro_batch_size=8,
)
acc_result = accum.compare_accumulation(net, ds, acfg, DEVICE)
rc, rw = acc_result["correct"], acc_result["wrong"]
gap = 100 * (rw.val_loss[-1] - rc.val_loss[-1]) / rc.val_loss[-1]
print(f"final val loss  correct : {rc.val_loss[-1]:.4f}")
print(f"final val loss  wrong   : {rw.val_loss[-1]:.4f}")
print(f"gap at the last eval    : {gap:+.2f}%")
path = accum.plot_accumulation(acc_result, "assets/accum_gap.png")
print("saved", path)
RESULTS["accum_final_val_correct"] = rc.val_loss[-1]
RESULTS["accum_final_val_wrong"] = rw.val_loss[-1]


In [ ]:
from IPython.display import Image
Image("assets/accum_gap.png")


The two curves start together and pull apart. The "wrong" rule is not random
noise — it systematically over-weights the short micro-batch's gradient, so it
optimises a slightly different objective than the one you think you wrote down.


## 4. Log the grad norm at every step, find where it moved first

The global grad norm is `sqrt(sum of squares of every gradient)`. We log it,
the raw loss, and an EMA-smoothed loss, every step. At one step we feed a single
corrupted batch (random targets). That batch has a large gradient → the norm
spikes **immediately**. The optimizer then takes one bad step, and the loss
only rises on the *next* normal batch.


In [ ]:
gcfg = gradnorm.GradNormConfig(
    n_steps=50 if SMOKE else 140,
    inject_step=30 if SMOKE else 70,
    batch_size=16,
    seq_len=min(128, cfg.block_size),
)
gtr = gradnorm.run(net, ds, gcfg, DEVICE)
lead = gradnorm.find_lead(gtr)
for k, v in lead.items():
    print(f"{k:>34} : {v}")
path = gradnorm.plot(gtr, lead, "assets/gradnorm_lead.png")
print("saved", path)
RESULTS["gradnorm_lead_steps"] = lead["grad norm led the loss by (steps)"]


In [ ]:
from IPython.display import Image
Image("assets/gradnorm_lead.png")


The grad norm spike lands one or more steps before the smoothed loss reacts.
That lead time is exactly why the grad norm is the most useful trace on a
training dashboard: a run about to diverge often shows it in the norm
thousands of steps before the loss curve bends.

Clipping (`clip_grad_norm_`) uses the same number — cap the norm, keep the
direction, scale down the length — and the notes say to have it on from step one.


## 5. Compute your own MFU

$$\text{MFU} = \frac{(6N + \text{attn}) \times \text{tokens per second}}{\text{peak FLOP/s of the accelerator}}$$

We time a real forward + backward + optimizer loop, count the tokens through
it, and divide by the vendor dense peak. Set `S10_PEAK_TFLOPS` to your card's
real number if the auto-guess is wrong.


In [ ]:
mcfg = mfu.MFUConfig(
    batch_size=16,
    seq_len=min(256, cfg.block_size),
    measure=10 if SMOKE else 30,
    dtype="fp32" if DEVICE.type == "cpu" else "bf16",
)
mres = mfu.measure_mfu(net, ds, mcfg, DEVICE)
print(mres.render())
RESULTS["mfu_pct"] = mres.mfu * 100
RESULTS["mfu_tokens_per_sec"] = mres.tokens_per_sec
RESULTS["mfu_achieved_tflops"] = mres.achieved_tflops


### Reporting it honestly — what is costing us the distance to 40%

This number will be **low**, and it should be. On this setup the gap to 40% is
paid for by, roughly in order:

1. **Tiny model.** `6N` is small, so each token is cheap in FLOPs but still pays
   full kernel-launch and Python-loop overhead. MFU rises with model size —
   this proxy is deliberately under the size where the GPU saturates.
2. **Small batch / short sequences.** The matmuls are memory-bandwidth bound,
   not compute bound. Bigger `B` and `T` amortise weight reads.
3. **No `torch.compile` / no fused kernels.** Eager mode launches hundreds of
   small kernels per step; attention is SDPA but the rest is not fused.
4. **fp32 on CPU / no tensor cores** in the smoke path — the peak column assumes
   bf16 tensor-core throughput the eager fp32 path never touches.
5. **Optimizer + `.item()` syncs.** AdamW is elementwise and bandwidth-bound;
   every `loss.item()` forces a device sync that stalls the pipeline.
6. **Attention term grows with `T²`** but is not counted in `6N`; at long
   context it eats real time that the `6N` estimate ignores, depressing MFU.

The fixes are the next three sessions: bigger models, bigger global batch via
gradient accumulation, `torch.compile`, activation checkpointing, and picking a
lower-precision format on purpose.


## 6. The number 0.1 in fp32, bf16 and fp8 E4M3

`0.1` is not exactly representable in binary — it is `0.0001100110011...`
repeating. Each format rounds that infinite tail at a different place.

**By hand.** `0.1 = 1.6 × 2⁻⁴`, so the unbiased exponent is `−4` in every
format below.

| format | sign | exponent (bias) | stored exp field | mantissa (implicit 1.) | value | rel. error |
| --- | --- | --- | --- | --- | --- | --- |
| **fp32** | 0 | −4 (127) → `123` = `01111011` | 8 bits | `1.10011001100110011001101₂` = 1.60000002 | 0.10000000149 | 1.5e-8 |
| **bf16** | 0 | −4 (127) → `123` = `01111011` | 8 bits | `1.1001101₂` = 1.6015625 | 0.10009765625 | 9.8e-4 |
| **fp8 E4M3** | 0 | −4 (7) → `3` = `0011` | 4 bits | `1.101₂` = 1.625 | 0.1015625 | 1.6e-2 |

Derivation of the mantissa bits (`0.6 × 2^m`, round to nearest):

- fp32: `0.6 × 2²³ = 5033164.8 → 5033165` → `10011001100110011001101`
- bf16: `0.6 × 2⁷  = 76.8 → 77`        → `1001101`
- fp8 : `0.6 × 2³  = 4.8 → 5`          → `101`

Now the same thing from the actual bits:


In [ ]:
print(floats.report(0.1))
RESULTS["float_0p1"] = {
    bd.fmt: {"bits": bd.bit_string, "value": bd.stored_value, "rel_err": bd.rel_error}
    for bd in floats.all_breakdowns(0.1)
}


### Which would I train in, and why

**bf16 — with an fp32 master copy of the weights** (the standard mixed-precision
recipe).

- **fp32** is the safe reference but doubles memory and bandwidth for detail
  training does not need. Used only for the master weights, the optimizer
  moments, and reductions (loss, norms).
- **fp16** has only 5 exponent bits; late-run gradients near `10⁻⁸` underflow to
  exactly zero and that weight silently stops learning. It needs loss scaling —
  one more thing to tune and get wrong.
- **bf16** keeps all 8 exponent bits, so its range equals fp32's and nothing in
  training underflows. It pays with precision — 2–3 decimal digits — which is
  fine for the forward/backward math because the fp32 master weights and fp32
  optimizer accumulate the small updates that bf16 alone would round away.
- **fp8 E4M3** (1.6% error on 0.1) only works blockwise, with per-block scales,
  attention kept in higher precision, and a handful of other tricks. It is a
  production recipe in 2026 but not the thing to *start* a from-scratch run in.

So: **bf16 for the model math, fp32 where it's cheap and matters.**


## 7. Sanity: the loop learns

Everything above is instrumentation. A short real run to confirm the five lines
actually drive the loss down, with grad-norm clipping on from step one as the
notes prescribe.


In [ ]:
import matplotlib.pyplot as plt

train_net = model_mod.GPT(cfg).to(DEVICE).train()
opt = torch.optim.AdamW(train_net.parameters(), lr=3e-3)
n_steps = 60 if SMOKE else 400
seq_len = min(128, cfg.block_size)
hist = {"step": [], "loss": [], "gnorm": []}
gen = torch.Generator().manual_seed(0)
t0 = time.perf_counter()
for step in range(n_steps):
    x, y = data.get_batch(ds, "train", 32, seq_len, DEVICE, generator=gen)
    opt.zero_grad(set_to_none=True)
    _, loss = train_net(x, targets=y)
    loss.backward()
    gnorm = torch.nn.utils.clip_grad_norm_(train_net.parameters(), 1.0)
    opt.step()
    if step % max(1, n_steps // 20) == 0 or step == n_steps - 1:
        hist["step"].append(step); hist["loss"].append(loss.item())
        hist["gnorm"].append(float(gnorm))
dt = time.perf_counter() - t0

print(f"start loss : {hist['loss'][0]:.3f}   (ln V = {math.log(ds.vocab_size):.3f})")
print(f"end   loss : {hist['loss'][-1]:.3f}")
print(f"wall time  : {dt:.1f}s for {n_steps} steps")
RESULTS["train_start_loss"] = hist["loss"][0]
RESULTS["train_end_loss"] = hist["loss"][-1]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(hist["step"], hist["loss"], "o-"); ax[0].set_title("train loss"); ax[0].set_xlabel("step"); ax[0].grid(alpha=0.3)
ax[0].axhline(math.log(ds.vocab_size), ls=":", color="k", label="ln V"); ax[0].legend()
ax[1].plot(hist["step"], hist["gnorm"], "o-", color="#cf222e"); ax[1].set_title("grad norm (pre-clip)"); ax[1].set_xlabel("step"); ax[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig("assets/train_curve.png", dpi=110, bbox_inches="tight"); plt.show()


In [ ]:
train_net.eval()
ctx = torch.tensor([[ds.stoi.get("\n", 0)]], dtype=torch.long, device=DEVICE)
out = train_net.generate(ctx, max_new_tokens=200, temperature=0.8)
print(ds.decode(out[0].tolist()))


## 8. The numbers, in one place


In [ ]:
print(json.dumps(RESULTS, indent=2, default=str))
with open("assets/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print("\nsaved assets/results.json")


### What this notebook checked

| # | claim | how it was checked |
| --- | --- | --- |
| 1 | every tensor in the step has a known shape and meaning | printed all three tables (activations, params, grads) |
| 2 | `backward()` computes the true `dL/dw` | central finite difference agreed to ≥6 decimals |
| 3 | average-of-averages accumulation is wrong with unequal micro-batches | 15.4% on the static case; a visible curve gap in a real run |
| 4 | the grad norm reacts before the loss | corrupted batch spiked the norm N steps before the smoothed loss |
| 5 | our MFU is low, and we know why | measured it; listed the six things between us and 40% |
| 6 | 0.1 is a different number in each float format | derived the bits by hand, confirmed against the raw bit patterns; train in bf16 + fp32 master |

**Every serious training bug is silent. Print things and check things.**
